# Chai-1 Structure Prediction

![Chai-1 Structure Prediction](https://proto-bio.github.io/proto-assets/images/tool/chai1/hero.png)

This notebook demonstrates multi-modal structure prediction using Chai-1 from Chai Discovery. Chai-1 predicts the three-dimensional structures of proteins, ligands, and glycans, as well as their complexes, by combining ESM language model embeddings with a trunk network and diffusion-based structure generation. We demonstrate `run_chai1` by predicting the structure of the MfnG protein bound to L-tyrosine, covering input construction, model configuration, result inspection, visualization, and export.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("chai1")
display_overview("chai1")
display_docs_section("chai1", "Background")

# Chai-1

First released in 2024, Chai-1 is [Chai Discovery](https://www.chaidiscovery.com/)'s multi-modal foundation model for molecular structure prediction, folding proteins together with small-molecule ligands and glycans. This toolkit runs Chai-1 to predict the joint 3D structure of protein, ligand, and glycan complexes from sequence, optionally conditioned on ESM embeddings and ColabFold multiple-sequence alignments.

Chai-1 ([Chai Discovery, 2024](https://doi.org/10.1101/2024.10.10.615955)) predicts the joint 3D structure of a biomolecular assembly from the sequences and chemical components it contains. It is a multi-modal foundation model that folds proteins together with small-molecule ligands, nucleic acids, glycans, and covalent modifications in a single model. Each protein chain can be conditioned on evolutionary signal, either through a multiple-sequence alignment (MSA) of related sequences or through embeddings from an ESM protein language model.

Internally, Chai-1 follows the all-atom co-folding approach popularized by AlphaFold3. It tokenizes the assembly the same way, with one token per amino-acid residue or nucleotide and one token per atom for ligands and modified residues. A trunk network then builds and refines token and pairwise representations, optionally conditioned on the MSA and ESM embeddings, and a diffusion module generates the all-atom coordinates by starting from noise and iteratively denoising into a structure. Several structures are sampled per input and ranked by an aggregate confidence score. Predicted confidence includes a per-atom predicted local distance difference test (pLDDT) for local reliability, a predicted aligned error (PAE) for the relative placement of any two tokens, and predicted template-modeling (pTM) and interface predicted template-modeling (ipTM) scores that summarize overall and interface accuracy.

The reference implementation is open-sourced by [Chai Discovery](https://www.chaidiscovery.com/) at [chaidiscovery/chai-lab](https://github.com/chaidiscovery/chai-lab), with both the code and the model weights released under the Apache-2.0 license for academic and commercial use, including drug discovery. Chai Discovery also runs the model as a hosted web platform at [lab.chaidiscovery.com](https://lab.chaidiscovery.com).

## Available tools

In [2]:
display_available_tools("chai1")

- **`run_chai1()`** — Multi-modal structure prediction using Chai1

### `run_chai1`

Chai-1 predicts 3D structures of proteins, ligands, and glycans using a diffusion-based model that optionally integrates ESM protein language model embeddings for improved predictions without requiring an explicit MSA search. The `num_trunk_recycles` parameter controls iterative refinement of the structure representation, while `num_diffn_timesteps` governs the resolution of the denoising process. Multiple independent diffusion samples can be generated via `num_diffn_samples`, with the best sample returned by confidence score. Chai-1 caps each complex at 2,048 tokens (1 per amino acid, 1 per heavy atom for ligands and glycans) and does not support DNA or RNA inputs.

In [3]:
from pathlib import Path

from proto_tools import (
    Chai1Config,
    Chai1Input,
    Chain,
    Complex,
    run_chai1,
)

In [4]:
# Display input docs
display_api_reference("chai1", "input", "run_chai1")

# MfnG protein sequence (enzyme from methanogenic archaea)
mfng_sequence = "MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDALYAYDVIDRIHDTNGFRYLLSAEARECLLPGTLFSLVGKFMHDINVAWPAWRNLAEVVRHGARDTSGAESPNGIAQEDYESLVGGINFWAPPIVTTLSRKLRASGRSGDATASVLDVGCGTGLYSQLLLREFPRWTATGLDVERIATLANAQALRLGVEERFATRAGDFWRGGWGTGYDLVLFANIFHLQTPASAVRLMRHAAACLAPDGLVAVVDQIVDADREPKTPQDRFALLFAASMTNTGGGDAYTFQEYEEWFTAAGLQRIETLDTPMHRILLARRATEPSAVPEGQASENLYFQ"

# L-tyrosine SMILES
tyrosine_smiles = "c1cc(ccc1C[C@@H](C(=O)O)N)O"

# Create protein-ligand complex
complex = Complex(
    chains=[
        Chain(sequence=mfng_sequence, entity_type="protein"),
        Chain(sequence=tyrosine_smiles, entity_type="ligand"),
    ]
)

# Create input
inputs = Chai1Input(complexes=[complex])

**Input** — `Chai1Input`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>complexes</code> | <code>list[proto_tools.entities.complex.Complex]</code> | required | List of complexes to predict structure for containing chains and entity types. |
| <code>msas</code> | <code>list[proto_tools.tools.structure_prediction.shared_data_models.ComplexMSAs] &#124; None</code> | <code>None</code> | Per-complex MSAs; a bare dict[int, MSA] is coerced to an unpaired ComplexMSAs. |

In [5]:
# Display config docs
display_api_reference("chai1", "config", "run_chai1")

# Configure Chai-1
config = Chai1Config(
    verbose=True,
    device="cuda",  # Change to "cpu" if no GPU available
    use_esm_embeddings=True,
    use_msa=False,
    num_trunk_recycles=3,
    num_diffn_timesteps=200,
    num_diffn_samples=1,
)

**Config** — `Chai1Config`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on (e.g., 'cuda', 'cpu') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>1200</code> | Maximum execution time in seconds |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>include_pae_matrix</code> | <code>bool</code> | <code>False</code> | Attach the full per-residue PAE matrix. |
| <code>use_msa</code> | <code>bool</code> | <code>True</code> | Whether to auto-generate MSAs via MMseqs2 homology search; supplied MSAs are always used |
| <code>msa_search_config</code> | <code>proto_tools.tools.sequence_alignment.mmseqs2.homology_search.Mmseqs2HomologySearchConfig &#124; None</code> | <code>None</code> | Nested MMseqs2 homology-search config for MSA generation; None uses default settings. |
| <code>pair_heterocomplex_msas</code> | <code>bool</code> | <code>True</code> | Whether heterocomplex protein chains should use taxonomy-paired MSA generation. |
| <code>use_esm_embeddings</code> | <code>bool</code> | <code>True</code> | Whether to use ESM embeddings for improved predictions |
| <code>num_trunk_recycles</code> | <code>int</code> | <code>3</code> | Iterative refinement passes through the trunk network. Higher = more accurate but slower. |
| <code>num_diffn_timesteps</code> | <code>int</code> | <code>200</code> | Denoising steps in the diffusion process. Higher = more refined but slower. |
| <code>num_diffn_samples</code> | <code>int</code> | <code>5</code> | Structure samples per complex; best by confidence is kept. Higher = more thorough but slower. |
| <code>num_trunk_samples</code> | <code>int</code> | <code>1</code> | Independent trunk forward passes per diffusion sample (adds sample diversity). |
| <code>low_memory</code> | <code>bool</code> | <code>True</code> | Stream features per sample to reduce peak GPU memory at the cost of speed. |
| <code>recycle_msa_subsample</code> | <code>int</code> | <code>0</code> | Randomly subsample the MSA across recycles for diversity. 0 disables. |

In [6]:
# Run structure prediction
result = run_chai1(inputs, config)

Folding structures (Chai-1):   0%|          | 0/1 [00:00<?, ?complex/s]

In [7]:
# Display output docs
display_api_reference("chai1", "output", "run_chai1")

mfng_structure = result.structures[0]

# Print confidence metrics
print(f"  Number of chains: {len(complex.chains)}")
print(f"  Protein length: {len(mfng_sequence)} residues")
print(f"  Average pLDDT: {mfng_structure.metrics.avg_plddt:.3f}")
print(f"  pTM score: {mfng_structure.metrics.ptm:.3f}")
print(f"  ipTM score: {mfng_structure.metrics.iptm:.3f}")

**Output** — `Chai1Output`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>structures</code> | <code>list[proto_tools.entities.structures.structure.Structure]</code> | required | List of predicted structures, one per input complex. |

  Number of chains: 2
  Protein length: 384 residues
  Average pLDDT: 0.838
  pTM score: 0.846
  ipTM score: 0.616


#### Visualize the predicted complex

The interactive viewer renders the predicted protein-ligand complex colored by chain. You can rotate, zoom, and inspect the binding interface between MfnG and L-tyrosine.

> NOTE: The 3D viewer below renders locally (JupyterLab, VS Code) but not in GitHub previews.

In [8]:
mfng_structure.visualize(color_by='chain')

## Export Results

Predicted structures can be exported to PDB or mmCIF format for further analysis in molecular visualization tools such as PyMOL, ChimeraX, or VMD. The B-factor column contains pLDDT confidence scores for per-residue visualization.

In [9]:
# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export results to PDB files
result.export(name="mfng_ligand_complex", export_path=output_dir, file_format="pdb")
print(f"Structure exported to {output_dir / 'mfng_ligand_complex.pdb'}")

Structure exported to example_output/mfng_ligand_complex.pdb


proto-tools/proto_tools/entities/structures/structure.py:661: UserWarning: CIF→PDB conversion: 1 residue name(s) exceed PDB's 3-character limit and will be truncated (e.g., ['LIG2']).
  return convert_cif_str_to_pdb_str(self.structure)
